# NYC Yellow Taxi Trip Data Analysis

**Project:** NYC Yellow Taxi Trip Data Analysis  
**Dataset:** `yellow_tripdata_2025-01.parquet`  
**Tools:** NumPy, Pandas, Matplotlib, Seaborn

## Business Objective
Analyze January 2025 NYC Yellow Taxi trip records to identify demand patterns, trip characteristics, passenger behavior, location patterns, payment types, vendors, rate codes, operational anomalies, and relationships between distance, duration and speed.

> **Important:** The assignment states that the dataset should not be treated as a revenue/profit/customer-spending dataset. This notebook therefore focuses only on fields represented in the assignment and does not make revenue or profit claims.


## 1. Problem Statement

The objective is to convert raw taxi-trip records into defensible business and operational insights through:

- Data understanding and quality assessment
- Data cleaning
- Feature engineering
- NumPy statistical analysis
- Exploratory data analysis
- Visualization
- Business-question analysis
- Key findings, limitations and conclusion

The analysis follows the mandatory project structure given in the assignment.


## 2. Dataset Description

The core analytical columns are:

- `VendorID`
- `tpep_pickup_datetime`
- `tpep_dropoff_datetime`
- `passenger_count`
- `trip_distance`
- `RatecodeID`
- `store_and_fwd_flag`
- `PULocationID`
- `DOLocationID`
- `payment_type`

`PULocationID` and `DOLocationID` are treated as Taxi Zone IDs, not neighborhood names, because no separate location lookup table was supplied with this project.


## 3. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid")

DATA_FILE = "yellow_tripdata_2025-01.parquet"

# Google Colab convenience: if the file is not already present, upload it.
if not Path(DATA_FILE).exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_FILE = next(iter(uploaded.keys()))
    except Exception:
        raise FileNotFoundError(
            f"Place {DATA_FILE} in the notebook working directory and rerun."
        )

print("Using:", DATA_FILE)


## 4. Load Dataset

In [ ]:
df = pd.read_parquet(DATA_FILE)

print("Shape:", df.shape)
display(df.head())
display(df.tail())


## 5. Initial Data Exploration

In [ ]:
print("Number of rows:", len(df))
print("Number of columns:", df.shape[1])

print("\nColumn names:")
display(pd.DataFrame({"column": df.columns}))

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nUnique values per column:")
display(df.nunique(dropna=False).to_frame("unique_values"))

print("\nMissing values:")
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100)
}).sort_values("missing_count", ascending=False)
display(missing)

print("\nDuplicate rows:", df.duplicated().sum())


## 6. Data Quality Assessment

In [ ]:
# Work on a copy for profiling
quality = df.copy()

# Datetime conversion for checking
pickup_dt = pd.to_datetime(quality["tpep_pickup_datetime"], errors="coerce")
dropoff_dt = pd.to_datetime(quality["tpep_dropoff_datetime"], errors="coerce")

quality_checks = {
    "pickup_datetime_invalid": int(pickup_dt.isna().sum()),
    "dropoff_datetime_invalid": int(dropoff_dt.isna().sum()),
    "dropoff_before_pickup": int((dropoff_dt < pickup_dt).sum()),
    "zero_duration": int(((dropoff_dt - pickup_dt).dt.total_seconds() == 0).sum()),
    "negative_duration": int(((dropoff_dt - pickup_dt).dt.total_seconds() < 0).sum()),
    "zero_or_negative_distance": int((quality["trip_distance"] <= 0).sum()),
    "negative_passenger_count": int((quality["passenger_count"] < 0).sum()),
    "missing_passenger_count": int(quality["passenger_count"].isna().sum())
}

display(pd.Series(quality_checks, name="count").to_frame())

print("VendorID values:")
display(sorted(quality["VendorID"].dropna().unique()))

print("RatecodeID values:")
display(sorted(quality["RatecodeID"].dropna().unique()))

print("store_and_fwd_flag values:")
display(sorted(quality["store_and_fwd_flag"].dropna().astype(str).unique()))

print("payment_type values:")
display(sorted(quality["payment_type"].dropna().unique()))

print("PULocationID range:", quality["PULocationID"].min(), "to", quality["PULocationID"].max())
print("DOLocationID range:", quality["DOLocationID"].min(), "to", quality["DOLocationID"].max())

print("\nTrip distance descriptive statistics:")
display(quality["trip_distance"].describe(percentiles=[.01,.05,.25,.5,.75,.90,.95,.99]).to_frame())


## 7. Data Cleaning

Cleaning decisions are deliberately conservative:

1. Convert pickup and drop-off timestamps to datetime.
2. Remove exact duplicate rows if present.
3. Remove rows with invalid/missing timestamps because trip duration cannot be calculated reliably.
4. Remove logically impossible negative-duration trips.
5. Remove negative trip distances.
6. Keep unusual positive distances/durations as potential outliers for investigation rather than automatically deleting them.
7. Keep missing passenger counts as missing rather than inventing a passenger number.


In [ ]:
clean_df = df.copy()

before = len(clean_df)

clean_df["tpep_pickup_datetime"] = pd.to_datetime(
    clean_df["tpep_pickup_datetime"], errors="coerce"
)
clean_df["tpep_dropoff_datetime"] = pd.to_datetime(
    clean_df["tpep_dropoff_datetime"], errors="coerce"
)

duplicate_count = int(clean_df.duplicated().sum())
clean_df = clean_df.drop_duplicates()

invalid_datetime = clean_df[
    clean_df["tpep_pickup_datetime"].isna() |
    clean_df["tpep_dropoff_datetime"].isna()
].shape[0]
clean_df = clean_df.dropna(
    subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"]
)

negative_duration = (
    clean_df["tpep_dropoff_datetime"] < clean_df["tpep_pickup_datetime"]
).sum()
clean_df = clean_df[
    clean_df["tpep_dropoff_datetime"] >= clean_df["tpep_pickup_datetime"]
]

negative_distance = (clean_df["trip_distance"] < 0).sum()
clean_df = clean_df[clean_df["trip_distance"] >= 0]

after = len(clean_df)

cleaning_summary = pd.DataFrame({
    "Metric": [
        "Original rows",
        "Exact duplicates identified",
        "Invalid datetime rows removed",
        "Negative-duration rows removed",
        "Negative-distance rows removed",
        "Final cleaned rows",
        "Rows removed overall"
    ],
    "Value": [
        before, duplicate_count, invalid_datetime,
        int(negative_duration), int(negative_distance),
        after, before-after
    ]
})
display(cleaning_summary)


## 8. Feature Engineering

Required analytical features:

- `trip_duration_minutes`
- `pickup_date`
- `pickup_hour`
- `day_of_week`
- `is_weekend`
- `average_speed_mph`
- `distance_category`
- `duration_category`
- `route`


In [ ]:
clean_df["trip_duration_minutes"] = (
    clean_df["tpep_dropoff_datetime"] -
    clean_df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

clean_df["pickup_date"] = clean_df["tpep_pickup_datetime"].dt.date
clean_df["pickup_hour"] = clean_df["tpep_pickup_datetime"].dt.hour
clean_df["day_of_week"] = clean_df["tpep_pickup_datetime"].dt.day_name()
clean_df["is_weekend"] = clean_df["tpep_pickup_datetime"].dt.dayofweek >= 5

duration_hours = clean_df["trip_duration_minutes"] / 60
clean_df["average_speed_mph"] = np.where(
    duration_hours > 0,
    clean_df["trip_distance"] / duration_hours,
    np.nan
)

# Clear, explainable business rules
clean_df["distance_category"] = pd.cut(
    clean_df["trip_distance"],
    bins=[-np.inf, 3, 10, np.inf],
    labels=["Short", "Medium", "Long"]
)

clean_df["duration_category"] = pd.cut(
    clean_df["trip_duration_minutes"],
    bins=[-np.inf, 5, 15, 30, np.inf],
    labels=["Very Short", "Short", "Medium", "Long"]
)

clean_df["route"] = (
    clean_df["PULocationID"].astype("Int64").astype(str)
    + " → " +
    clean_df["DOLocationID"].astype("Int64").astype(str)
)

display(clean_df.head())


## 9. NumPy Statistical Analysis

In [ ]:
numeric_vars = [
    "trip_distance",
    "trip_duration_minutes",
    "passenger_count",
    "average_speed_mph"
]

rows = []
for col in numeric_vars:
    s = pd.to_numeric(clean_df[col], errors="coerce").dropna()
    rows.append({
        "Variable": col,
        "Mean": np.mean(s),
        "Median": np.median(s),
        "Minimum": np.min(s),
        "Maximum": np.max(s),
        "Variance": np.var(s),
        "Std Dev": np.std(s),
        "P25": np.percentile(s, 25),
        "P50": np.percentile(s, 50),
        "P75": np.percentile(s, 75),
        "P90": np.percentile(s, 90),
        "P95": np.percentile(s, 95),
        "P99": np.percentile(s, 99)
    })

numpy_stats = pd.DataFrame(rows)
display(numpy_stats)


## 10. Outlier Analysis Using IQR

In [ ]:
def iqr_summary(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    q1, q3 = np.percentile(s, [25, 75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((s < lower) | (s > upper)).sum())
    return q1, q3, iqr, lower, upper, count

outlier_rows = []
for col in ["trip_distance", "trip_duration_minutes", "average_speed_mph"]:
    q1, q3, iqr, lower, upper, count = iqr_summary(clean_df[col])
    outlier_rows.append({
        "Variable": col,
        "Q1": q1, "Q3": q3, "IQR": iqr,
        "Lower Bound": lower, "Upper Bound": upper,
        "Potential Outliers": count
    })

outlier_df = pd.DataFrame(outlier_rows)
display(outlier_df)

print("Outliers are investigated rather than automatically deleted, following the assignment guidance.")


## 11. Time-Based Analysis

In [ ]:
hourly = clean_df.groupby("pickup_hour").agg(
    trips=("pickup_hour", "size"),
    avg_duration=("trip_duration_minutes", "mean"),
    avg_distance=("trip_distance", "mean"),
    avg_speed=("average_speed_mph", "mean")
).reset_index()

daily_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
daily = clean_df.groupby("day_of_week").size().reindex(daily_order).reset_index(name="trips")

weekday_weekend = clean_df.assign(
    period=np.where(clean_df["is_weekend"], "Weekend", "Weekday")
).groupby("period").size().reset_index(name="trips")

print("Busiest hour:", int(hourly.loc[hourly["trips"].idxmax(), "pickup_hour"]))
print("Lowest activity hour:", int(hourly.loc[hourly["trips"].idxmin(), "pickup_hour"]))
print("Busiest day:", daily.loc[daily["trips"].idxmax(), "day_of_week"])
print("Lowest activity day:", daily.loc[daily["trips"].idxmin(), "day_of_week"])

display(hourly)
display(daily)
display(weekday_weekend)


### Visualization 1 — Trips by Hour

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=hourly, x="pickup_hour", y="trips")
plt.title("NYC Yellow Taxi Trips by Pickup Hour")
plt.xlabel("Pickup Hour")
plt.ylabel("Number of Trips")
plt.tight_layout()
plt.show()

peak_hour = int(hourly.loc[hourly["trips"].idxmax(), "pickup_hour"])
low_hour = int(hourly.loc[hourly["trips"].idxmin(), "pickup_hour"])
print(f"Insight: Taxi activity peaks around hour {peak_hour} and is lowest around hour {low_hour}, showing that demand is not evenly distributed across the day.")


### Visualization 2 — Trips by Day of Week

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=daily, x="day_of_week", y="trips")
plt.title("NYC Yellow Taxi Trips by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Number of Trips")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(f"Insight: {daily.loc[daily['trips'].idxmax(), 'day_of_week']} has the highest recorded trip activity, while {daily.loc[daily['trips'].idxmin(), 'day_of_week']} has the lowest.")


### Visualization 3 — Hour × Day Demand Heatmap

In [ ]:
hour_day = pd.crosstab(clean_df["day_of_week"], clean_df["pickup_hour"]).reindex(daily_order)
plt.figure(figsize=(14,5))
sns.heatmap(hour_day, cmap="Blues")
plt.title("Taxi Demand Heatmap: Day of Week × Pickup Hour")
plt.xlabel("Pickup Hour")
plt.ylabel("Day of Week")
plt.tight_layout()
plt.show()

print("Insight: The heatmap identifies recurring high- and low-demand combinations of weekday/weekend and hour, which can support operational planning.")


### Visualization 4 — Average Duration by Hour

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=hourly, x="pickup_hour", y="avg_duration", marker="o")
plt.title("Average Trip Duration by Pickup Hour")
plt.xlabel("Pickup Hour")
plt.ylabel("Average Trip Duration (minutes)")
plt.tight_layout()
plt.show()

longest_hour = int(hourly.loc[hourly["avg_duration"].idxmax(), "pickup_hour"])
print(f"Insight: Hour {longest_hour} has the highest average trip duration in the cleaned data.")


### Visualization 5 — Average Distance by Hour

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=hourly, x="pickup_hour", y="avg_distance", marker="o")
plt.title("Average Trip Distance by Pickup Hour")
plt.xlabel("Pickup Hour")
plt.ylabel("Average Trip Distance (miles)")
plt.tight_layout()
plt.show()

far_hour = int(hourly.loc[hourly["avg_distance"].idxmax(), "pickup_hour"])
print(f"Insight: Hour {far_hour} has the highest average trip distance.")


## 12. Trip Distance & Duration Analysis

In [ ]:
trip_summary = clean_df[["trip_distance", "trip_duration_minutes"]].describe()
display(trip_summary)

corr_distance_duration = clean_df[["trip_distance", "trip_duration_minutes"]].corr().iloc[0,1]
print(f"Correlation between trip distance and duration: {corr_distance_duration:.3f}")


### Visualization 6 — Trip Distance Distribution

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(clean_df["trip_distance"], bins=80)
plt.xlim(0, clean_df["trip_distance"].quantile(.99))
plt.title("Distribution of Trip Distance")
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Number of Trips")
plt.tight_layout()
plt.show()

print("Insight: The histogram is limited to the 99th percentile for readability; extreme values remain in the data and are handled through outlier analysis.")


### Visualization 7 — Trip Duration Distribution

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(clean_df["trip_duration_minutes"], bins=80)
plt.xlim(0, clean_df["trip_duration_minutes"].quantile(.99))
plt.title("Distribution of Trip Duration")
plt.xlabel("Trip Duration (minutes)")
plt.ylabel("Number of Trips")
plt.tight_layout()
plt.show()

print("Insight: Most trips are concentrated within the lower duration range, while unusually long trips form a smaller tail that should be investigated rather than automatically deleted.")


### Visualization 8 — Distance Boxplot

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=clean_df["trip_distance"])
plt.xlim(clean_df["trip_distance"].quantile(.01), clean_df["trip_distance"].quantile(.99))
plt.title("Trip Distance Outlier View")
plt.xlabel("Trip Distance (miles)")
plt.tight_layout()
plt.show()

print("Insight: The boxplot highlights the central distribution and potential extreme-distance observations.")


### Visualization 9 — Duration Boxplot

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=clean_df["trip_duration_minutes"])
plt.xlim(clean_df["trip_duration_minutes"].quantile(.01), clean_df["trip_duration_minutes"].quantile(.99))
plt.title("Trip Duration Distribution and Outliers")
plt.xlabel("Trip Duration (minutes)")
plt.tight_layout()
plt.show()

print("Insight: Potentially extreme durations are visible as outliers; they require context before being classified as data errors.")


### Visualization 10 — Distance vs Duration

In [ ]:
sample_n = min(100000, len(clean_df))
scatter_df = clean_df[["trip_distance","trip_duration_minutes"]].dropna().sample(sample_n, random_state=42)

plt.figure(figsize=(10,6))
sns.scatterplot(data=scatter_df, x="trip_distance", y="trip_duration_minutes", alpha=.25, s=18)
plt.title("Trip Distance vs Trip Duration")
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Trip Duration (minutes)")
plt.xlim(0, scatter_df["trip_distance"].quantile(.99))
plt.ylim(0, scatter_df["trip_duration_minutes"].quantile(.99))
plt.tight_layout()
plt.show()

print(f"Insight: Distance and duration show a correlation of {corr_distance_duration:.3f}. Correlation describes association and does not prove causation.")


### Visualization 11 — Distance vs Average Speed

In [ ]:
speed_sample = clean_df[["trip_distance","average_speed_mph"]].dropna().sample(
    min(100000, clean_df["average_speed_mph"].notna().sum()), random_state=42
)

plt.figure(figsize=(10,6))
sns.scatterplot(data=speed_sample, x="trip_distance", y="average_speed_mph", alpha=.25, s=18)
plt.title("Trip Distance vs Average Speed")
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Average Speed (mph)")
plt.ylim(0, speed_sample["average_speed_mph"].quantile(.99))
plt.tight_layout()
plt.show()

print("Insight: Average speed can be strongly influenced by very short trips or unusual timestamps, so extreme speed values should be interpreted cautiously.")


## 13. Passenger Analysis

In [ ]:
passenger_counts = clean_df["passenger_count"].value_counts(dropna=False).sort_index()
passenger_pct = passenger_counts / passenger_counts.sum() * 100

print("Most common passenger count:", clean_df["passenger_count"].mode(dropna=True).iloc[0])
print("Percentage of trips with one passenger:",
      (clean_df["passenger_count"].eq(1).mean() * 100))

display(pd.DataFrame({
    "count": passenger_counts,
    "percent": passenger_pct
}))


### Visualization 12 — Passenger Count

In [ ]:
pc = clean_df["passenger_count"].dropna()
pc = pc[pc >= 0]

plt.figure(figsize=(9,5))
sns.countplot(x=pc)
plt.title("Passenger Count Distribution")
plt.xlabel("Passenger Count")
plt.ylabel("Number of Trips")
plt.tight_layout()
plt.show()

print("Insight: The passenger-count distribution is dominated by the most common recorded occupancy, while rare counts should be considered separately when interpreting trip behavior.")


### Passenger Count vs Trip Distance

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=clean_df, x="passenger_count", y="trip_distance")
plt.ylim(0, clean_df["trip_distance"].quantile(.99))
plt.title("Trip Distance by Passenger Count")
plt.xlabel("Passenger Count")
plt.ylabel("Trip Distance (miles)")
plt.tight_layout()
plt.show()


## 14. Pickup & Drop-off Location Analysis

In [ ]:
top_pickups = clean_df["PULocationID"].value_counts().head(10).reset_index()
top_pickups.columns = ["PULocationID", "trips"]

top_dropoffs = clean_df["DOLocationID"].value_counts().head(10).reset_index()
top_dropoffs.columns = ["DOLocationID", "trips"]

routes = (
    clean_df.groupby(["PULocationID","DOLocationID"])
    .size()
    .reset_index(name="trips")
    .sort_values("trips", ascending=False)
)

print("Top 10 pickup locations:")
display(top_pickups)

print("Top 10 drop-off locations:")
display(top_dropoffs)

print("Top 20 routes:")
display(routes.head(20))


### Visualization 13 — Top Pickup Locations

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=top_pickups, y="PULocationID", x="trips")
plt.title("Top 10 Pickup Location IDs")
plt.xlabel("Number of Trips")
plt.ylabel("Pickup Location ID")
plt.tight_layout()
plt.show()

print("Insight: These Taxi Zone IDs account for the highest pickup activity. They are IDs rather than neighborhood names because no zone lookup table was provided.")


### Visualization 14 — Top Drop-off Locations

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=top_dropoffs, y="DOLocationID", x="trips")
plt.title("Top 10 Drop-off Location IDs")
plt.xlabel("Number of Trips")
plt.ylabel("Drop-off Location ID")
plt.tight_layout()
plt.show()

print("Insight: These Taxi Zone IDs receive the highest number of recorded trips.")


## 15. Payment Type Analysis

In [ ]:
payment = clean_df["payment_type"].value_counts(dropna=False).reset_index()
payment.columns = ["payment_type", "trips"]
payment["percent"] = payment["trips"] / payment["trips"].sum() * 100
display(payment)


### Payment Type Visualization

In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=payment, x="payment_type", y="trips")
plt.title("Payment Type Distribution")
plt.xlabel("Payment Type Code")
plt.ylabel("Number of Trips")
plt.tight_layout()
plt.show()

print("Insight: The most common payment type is identified from recorded payment codes. The analysis does not interpret payment codes as revenue amounts.")


## 16. Vendor Analysis

In [ ]:
vendor = clean_df.groupby("VendorID").agg(
    trips=("VendorID","size"),
    avg_distance=("trip_distance","mean"),
    avg_duration=("trip_duration_minutes","mean"),
    avg_passengers=("passenger_count","mean")
).reset_index()
vendor["percent"] = vendor["trips"] / vendor["trips"].sum() * 100
display(vendor)


### Vendor Visualization

In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=vendor, x="VendorID", y="trips")
plt.title("Trips by Vendor")
plt.xlabel("Vendor ID")
plt.ylabel("Number of Trips")
plt.tight_layout()
plt.show()

print("Insight: Vendor differences are descriptive only. The dataset does not provide an appropriate performance measure for declaring one vendor better than another.")


## 17. Rate Code Analysis

In [ ]:
rate = clean_df.groupby("RatecodeID").agg(
    trips=("RatecodeID","size"),
    avg_distance=("trip_distance","mean"),
    avg_duration=("trip_duration_minutes","mean")
).reset_index()
rate["percent"] = rate["trips"] / rate["trips"].sum() * 100
display(rate.sort_values("trips", ascending=False))


## 18. Store & Forward Analysis

In [ ]:
sf = clean_df.groupby("store_and_fwd_flag").agg(
    trips=("store_and_fwd_flag","size"),
    avg_distance=("trip_distance","mean"),
    avg_duration=("trip_duration_minutes","mean")
).reset_index()
sf["percent"] = sf["trips"] / sf["trips"].sum() * 100
display(sf)


## 19. Correlation Analysis

In [ ]:
corr_cols = [
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "average_speed_mph"
]
corr = clean_df[corr_cols].corr()
display(corr)

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

print("Insight: Correlation measures association between variables. It does not establish causation.")


## 20. Advanced Analysis — Peak vs Off-Peak

For a simple operational comparison, peak hours are defined as **07:00–09:59 and 16:00–18:59**. All other hours are treated as off-peak.


In [ ]:
peak_hours = [7,8,9,16,17,18]
clean_df["peak_period"] = np.where(
    clean_df["pickup_hour"].isin(peak_hours), "Peak", "Off-Peak"
)

peak_summary = clean_df.groupby("peak_period").agg(
    trips=("peak_period","size"),
    avg_distance=("trip_distance","mean"),
    avg_duration=("trip_duration_minutes","mean"),
    avg_speed=("average_speed_mph","mean")
).reset_index()

display(peak_summary)


## 21. Route Analysis

In [ ]:
top_routes = (
    clean_df.groupby(["PULocationID","DOLocationID"])
    .agg(
        trips=("route","size"),
        avg_distance=("trip_distance","mean"),
        avg_duration=("trip_duration_minutes","mean")
    )
    .reset_index()
    .sort_values("trips", ascending=False)
)

display(top_routes.head(20))


## 22. Final KPI Summary

In [ ]:
most_common_passenger = clean_df["passenger_count"].mode(dropna=True).iloc[0]
busiest_hour = int(hourly.loc[hourly["trips"].idxmax(), "pickup_hour"])
busiest_day = daily.loc[daily["trips"].idxmax(), "day_of_week"]
most_common_payment = clean_df["payment_type"].mode(dropna=True).iloc[0]
most_common_pickup = clean_df["PULocationID"].mode(dropna=True).iloc[0]
most_common_dropoff = clean_df["DOLocationID"].mode(dropna=True).iloc[0]
most_common_route = routes.iloc[0]

kpi = pd.DataFrame({
    "KPI": [
        "Total Trips",
        "Average Trip Distance",
        "Median Trip Distance",
        "Average Trip Duration",
        "Median Trip Duration",
        "Maximum Trip Distance",
        "Maximum Trip Duration",
        "Average Speed",
        "Most Common Passenger Count",
        "Busiest Pickup Hour",
        "Busiest Day",
        "Most Common Payment Type",
        "Most Common Pickup Location",
        "Most Common Drop-off Location",
        "Most Common Route"
    ],
    "Value": [
        len(clean_df),
        clean_df["trip_distance"].mean(),
        clean_df["trip_distance"].median(),
        clean_df["trip_duration_minutes"].mean(),
        clean_df["trip_duration_minutes"].median(),
        clean_df["trip_distance"].max(),
        clean_df["trip_duration_minutes"].max(),
        clean_df["average_speed_mph"].mean(),
        most_common_passenger,
        busiest_hour,
        busiest_day,
        most_common_payment,
        most_common_pickup,
        most_common_dropoff,
        f"{most_common_route['PULocationID']} → {most_common_route['DOLocationID']}"
    ]
})
display(kpi)


## 23. Business Questions — Executive Summary

In [ ]:
print("1. When is taxi demand highest?")
print(f"   Highest pickup activity occurs at hour {busiest_hour}.")

print("\n2. When is taxi demand lowest?")
print(f"   Lowest pickup activity occurs at hour {int(hourly.loc[hourly['trips'].idxmin(), 'pickup_hour'])}.")

print("\n3. Weekday vs weekend demand:")
display(weekday_weekend)

print("\n4. Typical trip:")
print(f"   Average distance = {clean_df['trip_distance'].mean():.2f} miles; "
      f"median distance = {clean_df['trip_distance'].median():.2f} miles; "
      f"average duration = {clean_df['trip_duration_minutes'].mean():.2f} minutes; "
      f"median duration = {clean_df['trip_duration_minutes'].median():.2f} minutes.")

print("\n5. Most common pickup location ID:", most_common_pickup)
print("6. Most common drop-off location ID:", most_common_dropoff)
print("7. Most frequent route:", f"{most_common_route['PULocationID']} → {most_common_route['DOLocationID']}")
print("8. Distance-duration relationship:", f"correlation = {corr_distance_duration:.3f}")
print("9. Most common passenger count:", most_common_passenger)
print("10. Most common payment type code:", most_common_payment)
print("11. Most frequently used rate code:", clean_df["RatecodeID"].mode(dropna=True).iloc[0])


## 24. Key Findings

After running the notebook, use the generated KPI table, charts and statistical tables to report the strongest evidence-based findings.

The analysis should focus on:

- Demand concentration by hour and day
- Weekday versus weekend activity
- Typical trip distance and duration
- Distance-duration relationship
- Passenger-count distribution
- Top pickup and drop-off Taxi Zone IDs
- Most frequent routes
- Payment-type distribution
- Vendor and rate-code differences
- Store-and-forward differences
- Potential data-quality anomalies and outliers


## 25. Business / Operational Insights

Potential operational uses of this analysis include:

1. Aligning taxi availability with high-demand hours.
2. Comparing weekday and weekend demand patterns.
3. Identifying heavily used pickup and drop-off zones.
4. Monitoring common routes for operational planning.
5. Investigating unusually long trips, extreme distances or abnormal speeds.
6. Comparing descriptive trip characteristics across vendors and rate codes.
7. Monitoring payment-type and store-and-forward distributions.

These are operational observations, not causal claims.


## 26. Limitations

- The assignment does not provide a Taxi Zone lookup table, so location IDs are not converted into neighborhood names.
- The project should not claim revenue, profit, driver earnings or customer spending because those fields are outside the requested analytical scope.
- Outliers are not automatically errors; they require investigation.
- Correlation does not imply causation.
- Rare categories may have small sample sizes and should be interpreted cautiously.


## 27. Final Conclusion

This project demonstrates an end-to-end Python EDA workflow using Pandas, NumPy, Matplotlib and Seaborn. It loads the January 2025 NYC Yellow Taxi data, assesses data quality, creates analytical features, performs statistical analysis, investigates time/location/passenger/payment/vendor/rate-code patterns, identifies potential outliers, visualizes important relationships and summarizes evidence-based operational insights.

Run all cells from top to bottom before submission so that the notebook contains the actual outputs and visualizations required by the assignment.


## 28. Export Cleaned Dataset

Run this final cell to save the cleaned/derived dataset as a CSV if your instructor wants a separate cleaned dataset.

In [ ]:
OUTPUT_CSV = "NYC_Yellow_Taxi_Cleaned_Jan2025.csv"
clean_df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)
